## Proyecto GT - Fase 3 - Preparación
 Data preparation



**Objetivos:** imputación de faltantes, tratamiento de outliers, creación de variables y escalamiento.

In [ ]:
#Importación de librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import datetime
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from kedro.framework.session import KedroSession

session = KedroSession.create()
context = session.load_context()
datos = context.catalog.load("datos_crudos")


In [ ]:
# Carga del dataset (asegúrate de tener el CSV en la misma carpeta que este notebook)
csv_path = "gaming.csv"
assert os.path.exists(csv_path), "No se encontró el archivo Gaming-Trends-2024.csv en el directorio actual."

df = pd.read_csv(csv_path)
print("Shape:", df.shape)
print("Columnas:", list(df.columns))
df.head()


In [ ]:

def find_col(df, key_candidates):
    # Try to find a column in df whose lowercased name contains ALL tokens in any candidate string.
    # Returns the best match or raises ValueError if not found.
    cols = list(df.columns)
    lower = [c.lower() for c in cols]
    scores = []
    for cand in key_candidates:
        tokens = [t.strip() for t in cand.lower().replace("(", " ").replace(")", " ").replace("$"," ").split() if t.strip()]
        for i, lc in enumerate(lower):
            if all(tok in lc for tok in tokens):
                scores.append((i, cols[i], len(tokens)))
                print(f"find_col: candidate '{cand}' matches column '{cols[i]}'")
    if not scores:
        raise ValueError(f"No se encontró ninguna columna para {key_candidates}")
    # prefer the match with more tokens (more specific)
    scores.sort(key=lambda x: (-x[2], x[0]))
    return scores[0][1]

In [ ]:
df_prep = df.copy()

# Imputación simple
for col in df_prep.columns:
    if df_prep[col].dtype.kind in "biufc":
        med = df_prep[col].median()
        df_prep[col] = df_prep[col].fillna(med)
    else:
        mode_val = df_prep[col].mode().iloc[0] if not df_prep[col].mode().empty else "Desconocido"
        df_prep[col] = df_prep[col].fillna(mode_val)

# Clipping por percentiles para estabilizar extremos en numéricas
num_cols = df_prep.select_dtypes(include=[np.number]).columns
lower = df_prep[num_cols].quantile(0.01)
upper = df_prep[num_cols].quantile(0.99)
df_prep[num_cols] = df_prep[num_cols].clip(lower=lower, upper=upper, axis=1)

# Métricas derivadas (si están disponibles)
try:
    dau_col = find_col(df_prep, ["daily active users", "dau"])
    revenue_col = find_col(df_prep, ["revenue", "ingresos"])
    df_prep["ARPU"] = df_prep[revenue_col] / df_prep[dau_col].replace(0, np.nan)
except Exception:
    df_prep["ARPU"] = np.nan

# Variables categóricas A one-hot encoding
cat_cols = df_prep.select_dtypes(include=["object", "category"]).columns.tolist()
# excluir columna de fecha en texto si existiera
try:
    maybe_date = find_col(df_prep, ["date","fecha"])
    cat_cols = [c for c in cat_cols if c != maybe_date]
except Exception:
    pass

df_model = pd.get_dummies(df_prep, columns=cat_cols, drop_first=True)

print("Shape df_model:", df_model.shape)
df_model.head()

In [ ]:
col = 'Revenue'  # columna objetivo

# Asegurar formato de fecha
df["Date"] = pd.to_datetime(df["Date"])

# Agrupar por mes y calcular la media solo en columnas numéricas
df_mensual = df.resample('M', on='Date')[df.select_dtypes(include=np.number).columns].mean().reset_index()

# Crear columna de días desde la primera fecha
df_mensual["dias"] = (df_mensual["Date"] - df_mensual["Date"].min()).dt.days

# Variables
X = df_mensual[["dias"]]
y = df_mensual[col]

# Modelo
modelo = LinearRegression()
modelo.fit(X, y)
df_mensual["prediccion"] = modelo.predict(X)

# Gráfico
plt.figure(figsize=(10,6))
plt.plot(df_mensual["Date"], y, label="Datos reales")
plt.plot(df_mensual["Date"], df_mensual["prediccion"], label="Regresión lineal", color="red")
plt.xlabel("Fecha")
plt.ylabel(col)
plt.title("Regresión lineal mensual")
plt.legend()
plt.grid(True)
#plt.savefig('graficos_generados/regresion_mensual.png')
plt.show()


In [ ]:
#cambio de columna objetivo
col = 'New Registrations'  # columna objetivo

# Histograma
plt.figure(figsize=(10,6))
sns.histplot(df[col], bins=30, kde=True, color='skyblue') # histograma con línea KDE
plt.title('Histograma de ' + col)
plt.xlabel(col)
plt.ylabel('Frecuencia')
#plt.savefig('graficos_generados/histograma registros.png')
plt.show()

In [ ]:
# Agrupar por año
df_anual = df.resample('Y', on='Date').agg({col: list})

# Crear boxplot por año
plt.figure(figsize=(12,6))
plt.boxplot(df_anual[col], labels=df_anual.index.year)
plt.title(f"Distribución anual de {col} ($)")
plt.xlabel("Año")
plt.ylabel(col)
plt.grid(True)
#plt.savefig('graficos_generados/boxplot_anual.png')
plt.show()


## Limpieza

In [ ]:
df_cleaned = df.drop_duplicates()

df_cleaned = df_cleaned.dropna()

df_cleaned.shape, df_cleaned.isna().sum().sum(), df_cleaned.duplicated().sum()

In [ ]:
# Asegurar que la columna existe y no tiene valores nulos
df['Top Genre'] = df['Top Genre'].fillna('Desconocido')

# Extraer géneros únicos
generos_unicos = df['Top Genre'].unique()

# Mostrar ordenados
print(" Géneros únicos encontrados:")
for genero in sorted(generos_unicos):
    print("-", genero)

In [ ]:
csv_path = "gaming.csv"
assert os.path.exists(csv_path), "No se encontró el archivo Gaming-Trends-2024.csv en el directorio actual."

df = pd.read_csv(csv_path)
print("Shape:", df.shape)
print("Columnas:", list(df.columns))
df.head()

genre_mapping = {
    'Action': '00',
    'Simulation': '01',
    'RPG': '02',
    'FPS': '03',
    'Adventure': '04'
}

df['Top Genre'] = df['Top Genre'].replace(genre_mapping)

df.head()

In [ ]:
#Date a tipo datetime
#Extraer componentes: año, mes, día, día de la semana, trimestre.

df["Date"] = pd.to_datetime(df["Date"])
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Weekday"] = df["Date"].dt.day_name()


In [ ]:
#Normalización y escalado

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df[["DAU", "Revenue ($)", "In-game Purchases ($)"]] = scaler.fit_transform(df[["DAU", "Revenue ($)", "In-game Purchases ($)"]])


In [ ]:
#Ratio de compras por usuario
df["Purchases_per_user"] = df["In-game Purchases ($)"] / df["DAU"]

#Ratio de compras por usuario:
df["Purchases_per_user"] = df["In-game Purchases ($)"] / df["DAU"]

#Engagement estimado: duración × usuarios activos
df["Engagement"] = df["Session Duration (minutes)"] * df["DAU"]

#Influencia social: menciones + endorsements
df["Social_Impact"] = df["Social Media Mentions"] + df["Influencer Endorsements"]


